#**⚠️ Prerequisites**  
Before running any cell, enable GPU runtime:
1. Go to **Runtime** → **Change runtime type**
2. Set **Hardware accelerator** to **T4 GPU**
3. Click **Save**
4. Then run the cells top to bottom.

## Step 1 — Verify GPU is available

In [8]:
# Verify GPU runtime and CUDA availability
!nvidia-smi
!nvcc --version

Wed Apr 29 13:26:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## Program 1 — CUDA Vector Addition

In [9]:
%%writefile vector_add.cu
#include <iostream>
#include <cuda_runtime.h>
using namespace std;

__global__ void vectorAdd(int* A, int* B, int* C, int size) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < size) {
        C[tid] = A[tid] + B[tid];
    }
}

void print(int* vector, int size) {
    for (int i = 0; i < size; i++) {
        cout << vector[i] << " ";
    }
    cout << endl;
}

int main() {
    int N = 1000000;
    size_t bytes = N * sizeof(int);

    // Allocate host memory
    int* h_A = new int[N];
    int* h_B = new int[N];
    int* h_C = new int[N];

    // Initialize vectors
    for (int i = 0; i < N; i++) {
        h_A[i] = i;
        h_B[i] = i * 2;
    }

    cout << "Vector Size: " << N << endl << endl;
    cout << "First 5 elements of A: ";
    for(int i = 0; i < 5; i++) cout << h_A[i] << " ";
    cout << endl;

    cout << "First 5 elements of B: ";
    for(int i = 0; i < 5; i++) cout << h_B[i] << " ";
    cout << endl << endl;

    // Allocate device memory
    int* d_A, * d_B, * d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    // Copy data to device
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    // Launch kernel
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    cout << "Threads per block: " << threadsPerBlock << endl;
    cout << "Blocks per grid: " << blocksPerGrid << endl << endl;

    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    // Copy result back to host
    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);

    cout << "First 5 elements of C (A+B): ";
    for(int i = 0; i < 5; i++) cout << h_C[i] << " ";
    cout << endl << endl;

    // Verify result
    bool correct = true;
    for (int i = 0; i < N; i++) {
        if (h_C[i] != h_A[i] + h_B[i]) {
            correct = false;
            break;
        }
    }
    cout << "Result: " << (correct ? "CORRECT" : "INCORRECT") << endl;

    // Cleanup
    delete[] h_A; delete[] h_B; delete[] h_C;
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);

    return 0;
}

Overwriting vector_add.cu


In [10]:
# Compile Vector Addition
!nvcc -o vector_add vector_add.cu
print("Compilation successful!")

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Compilation successful!


In [11]:
# Run Vector Addition
!./vector_add

Vector Size: 1000000

First 5 elements of A: 0 1 2 3 4 
First 5 elements of B: 0 2 4 6 8 

Threads per block: 256
Blocks per grid: 3907

First 5 elements of C (A+B): 0 3 6 9 12 

Result: CORRECT


---
## Program 2 — CUDA Matrix Multiplication

In [12]:
%%writefile matrix_mul.cu
#include <iostream>
#include <cuda_runtime.h>
using namespace std;

__global__ void matrixMultiply(int* A, int* B, int* C, int size) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < size && col < size) {
        int sum = 0;
        for (int i = 0; i < size; i++) {
            sum += A[row * size + i] * B[i * size + col];
        }
        C[row * size + col] = sum;
    }
}

void initialize(int* matrix, int size) {
    for (int i = 0; i < size * size; i++) {
        matrix[i] = rand() % 10;
    }
}

void print(int* matrix, int size) {
    for (int row = 0; row < size; row++) {
        for (int col = 0; col < size; col++) {
            cout << matrix[row * size + col] << " ";
        }
        cout << endl;
    }
    cout << endl;
}

int main() {
    int N = 512;  // Matrix size NxN
    size_t bytes = N * N * sizeof(int);

    // Allocate host memory
    int* h_A = new int[N * N];
    int* h_B = new int[N * N];
    int* h_C = new int[N * N];

    // Initialize matrices
    initialize(h_A, N);
    initialize(h_B, N);

    cout << "Matrix Size: " << N << "x" << N << endl << endl;

    // Print sample values (only for small matrices)
    if (N <= 4) {
        cout << "Matrix A:\n"; print(h_A, N);
        cout << "Matrix B:\n"; print(h_B, N);
    }

    // Allocate device memory
    int* d_A, * d_B, * d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    // Copy data to device
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    // Launch kernel
    int THREADS = 16;
    int BLOCKS = (N + THREADS - 1) / THREADS;

    dim3 threads(THREADS, THREADS);
    dim3 blocks(BLOCKS, BLOCKS);

    cout << "Grid: " << BLOCKS << "x" << BLOCKS << " blocks" << endl;
    cout << "Threads: " << THREADS << "x" << THREADS << " per block" << endl << endl;

    matrixMultiply<<<blocks, threads>>>(d_A, d_B, d_C, N);

    // Copy result back to host
    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);

    if (N <= 4) {
        cout << "Result Matrix C (A x B):\n";
        print(h_C, N);
    } else {
        cout << "First 5 elements of result: ";
        for(int i = 0; i < 5; i++) cout << h_C[i] << " ";
        cout << endl << endl;
    }

    cout << "Computation complete!" << endl;

    // Cleanup
    delete[] h_A; delete[] h_B; delete[] h_C;
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);

    return 0;
}

Overwriting matrix_mul.cu


In [13]:
# Compile Matrix Multiplication
!nvcc -o matrix_mul matrix_mul.cu
print("Compilation successful!")

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Compilation successful!


In [14]:
# Run Matrix Multiplication
!./matrix_mul

Matrix Size: 512x512

Grid: 32x32 blocks
Threads: 16x16 per block

First 5 elements of result: 11057 10652 10663 10558 10841 

Computation complete!
